# Generate filter phrases to be used in the ESRI Online Assistant.   
Creating a filter from all counties in Thrive Region and matching them with the states.  
AGOL Assistant (?) might be the name for the tool.

In [1]:
import pandas as pd
import geopandas as gpd

path = r"C:\Users\cansu\Dropbox\studio2263\Thrive\GIS\base_map\thrive_boundaries\thrive_counties.geojson"

In [2]:
counties = gpd.read_file( path )
counties.head()

,STATEFP,COUNTYFP,COUNTYNS,GEOID,GEOIDFQ,NAME,NAMELSAD,LSAD,CLASSFP,thrive_cty,geometry
0,13,295,00354216,13295,0500000US13295,Walker,Walker County,06,H1,primary,"MULTIPOLYGON (((-85.26436 34.88109, -85.26439 ..."
1,01,095,00161574,01095,0500000US01095,Marshall,Marshall County,06,H1,secondary,"MULTIPOLYGON (((-86.14981 34.53363, -86.14864 ..."
2,13,055,00352213,13055,0500000US13055,Chattooga,Chattooga County,06,H1,secondary,"MULTIPOLYGON (((-85.34672 34.35625, -85.34898 ..."
3,47,139,01639782,47139,0500000US47139,Polk,Polk County,06,H1,primary,"MULTIPOLYGON (((-84.71598 35.2329, -84.71582 3..."
4,47,145,01639785,47145,0500000US47145,Roane,Roane County,06,H1,secondary,"MULTIPOLYGON (((-84.44733 35.75908, -84.44741 ..."


Convert `statefp`'s to State Names, the Living Atlas data only contains the State Names.

In [9]:
states = {
    "01": 'Alabama',
    "13": 'Georgia',
    '47': 'Tennessee'
}

states = pd.DataFrame(list(states.items()), columns=['statefp', 'state name'])
states

,statefp,state name
0,01,Alabama
1,13,Georgia
2,47,Tennessee


In [13]:
counties_joined = counties.join( states.set_index('statefp'), on="STATEFP" )

grouped = counties_joined.groupby("state name").agg({
    "NAMELSAD": list
})

grouped

,NAMELSAD
state name,
Alabama,"[Marshall County, Jackson County, Madison Coun..."
Georgia,"[Walker County, Chattooga County, Floyd County..."
Tennessee,"[Polk County, Roane County, Cumberland County,..."


## Iterate over States in a group to iterate over its counties. 

In [40]:
all_states = []
for i,r in grouped.iterrows():

    c_query = []
    for count, line in enumerate(r.values[0]):
        if count < len( r.values[0] )-1 :
            a = f"County = '{line}' or"
        else:
            a = f"County = '{line}'"
        c_query.append(a)

    c_query = " ".join( c_query )
    c_query = f"({c_query})"
    # print( c_query )

    text = f"(State = '{i}' AND {c_query})"
    all_states.append( text )


" OR ".join( all_states)

"(State = 'Alabama' AND (County = 'Marshall County' or County = 'Jackson County' or County = 'Madison County' or County = 'DeKalb County' or County = 'Cherokee County' or County = 'Etowah County')) OR (State = 'Georgia' AND (County = 'Walker County' or County = 'Chattooga County' or County = 'Floyd County' or County = 'Murray County' or County = 'Dade County' or County = 'Fannin County' or County = 'Gordon County' or County = 'Catoosa County' or County = 'Gilmer County' or County = 'Whitfield County')) OR (State = 'Tennessee' AND (County = 'Polk County' or County = 'Roane County' or County = 'Cumberland County' or County = 'Van Buren County' or County = 'Loudon County' or County = 'Rhea County' or County = 'Monroe County' or County = 'Sequatchie County' or County = 'Warren County' or County = 'Meigs County' or County = 'Franklin County' or County = 'Bledsoe County' or County = 'McMinn County' or County = 'Marion County' or County = 'Grundy County' or County = 'Bradley County' or County